# Diagnóstico dos fretes
Execute previamente `python scripts/build_data.py` e `python scripts/analyze.py`. Valores agregados usam peso como ponderador. Agosto/2026 é parcial.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
from frete.modeling import read_data
df = read_data(ROOT / 'data/processed/base_modelagem_frete.csv')
display(df.head())
display(df[['frete_rs_t', 'peso_liquido_t', 'Valor líquido']].describe(percentiles=[.01,.5,.99]))

In [ ]:
display(df.filter(like='flag_').sum().to_frame('ocorrencias'))
by = df.groupby('destino_frete').agg(valor=('Valor líquido','sum'), peso=('peso_liquido_t','sum'), itens=('Item','size'))
by['tarifa_ponderada_rs_t'] = by.valor / by.peso
display(by.sort_values('valor', ascending=False))

## Reprodução
O processamento e as regras estão em `src/frete/data.py`; as análises e gráficos são reconstruídos por `scripts/analyze.py`. Conferir a auditoria antes de interpretar valores extremos. Destino cadastrado pelo representante não comprova o itinerário real.

## Referência pública de diesel
A série mensal ANP é contexto retrospectivo, não preço pago pela transportadora nem atributo do modelo. Agosto completo difere do histórico operacional encerrado em 04/08.

In [ ]:
diesel_path = ROOT / 'data/external/diesel.csv'
if diesel_path.exists():
    diesel = pd.read_csv(diesel_path)
    display(diesel.head())
    if not diesel.empty:
        display(diesel.groupby(['localidade', 'produto']).agg(meses=('data','nunique'), preco_min=('preco_rs_l','min'), preco_max=('preco_rs_l','max')))